# Game Retention — bắt đầu ở đây
**GIẢ LẬP seed 42**, không đại diện game thật. Shift+Enter từ trên xuống.
Câu hỏi: một người có nhiều phiên sẽ ảnh hưởng COUNT như thế nào? Dự đoán trước khi chạy.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'game_lab').is_dir() and root!=root.parent: root=root.parent
if not (root/'game_lab').is_dir(): raise RuntimeError('Mở bằng python run_game.py --notebook tại thư mục dự án')
sys.path.insert(0,str(root))
from game_lab.learning import read_sql,learning_db
db=learning_db()
print('Database chỉ đọc:',db)


## Đọc nguồn và hiểu mỗi dòng

In [ ]:
read_sql('SELECT * FROM metadata')

In [ ]:
read_sql('SELECT COUNT(*) AS sessions,COUNT(DISTINCT player_id) AS players FROM sessions')

## Đếm đúng cấp người–ngày
So session_count và active_players; tìm ngày có chênh lệch.

In [ ]:
read_sql('SELECT date(started_at) AS activity_date,COUNT(*) AS session_count,COUNT(DISTINCT player_id) AS active_players FROM sessions GROUP BY date(started_at) ORDER BY activity_date;')

## Nhóm ngày chơi đầu
MIN trên toàn lịch sử rồi mới lọc nhóm.

In [ ]:
read_sql('WITH firsts AS (SELECT player_id,MIN(date(started_at)) AS cohort_date FROM sessions GROUP BY player_id) SELECT cohort_date,COUNT(*) AS players FROM firsts GROUP BY cohort_date ORDER BY cohort_date;')

## Retention và thời gian quan sát
SQL dựng người–ngày trước, chỉ dùng mẫu đã trưởng thành. So D1/D7/D30 và xem ô cuối tháng không thể tính D30.

In [ ]:
from game_lab.analysis import sql_tables,compare
tables=sql_tables(db)
tables['retention']

In [ ]:
tables['cohorts'].tail(31)

## Tiến độ màn
Số lượt thử khác số người; trung vị chỉ dùng người đã hoàn thành.

In [ ]:
tables['levels']

## Đối chiếu pandas độc lập
Nguồn chung, cách tổng hợp độc lập. Sai thì dừng trước khi xuất báo cáo.

In [ ]:
tables=compare(db)
assert tables['daily'].active_players.sum()==len(tables['player_days'])
print('SQL/pandas và tổng người–ngày khớp')

## Bốn hình và ba nhận xét
Mẫu số từng Dn có thể khác. Ô chưa đủ quan sát giữ trống, không tô thành 0%.

In [ ]:
from game_lab.analyze import export
from IPython.display import display,Image,Markdown
export(db,out=root/'outputs/notebook')
for path in sorted((root/'outputs/notebook').glob('*.png')): display(Image(filename=str(path)))
display(Markdown((root/'outputs/notebook/report.md').read_text(encoding='utf-8')))

## Tự làm
Chọn nhóm tháng 1, quan sát hết tháng 3, giải thích vì sao không cắt sự kiện tại 31/01. Viết ba nhận xét có nhóm, mốc và mẫu số; không kết luận chưa qua màn = bỏ game.